# 02. Térbeli mérések

## MIT mérünk?

Ez a fejezet a lakások **térbeli helyzetét** méri három rétegben: (1) a vasúti vágánytól mért légvonali távolságból képzett **immissziós sávok** (a zaj- és rezgésterhelés kategóriái), (2) az úthálózaton mért gyalogos **izokrónok** (hány perc séta a legközelebbi állomás/megálló: 5-10-15 perces gyűrűk), és (3) a gyalogosan elérhető szolgáltatások (**POI-k**) száma ugyanezen a 15 perces körzeten belül. Emellett az árak **térbeli eloszlását** ábrázoljuk: ponttérkép, interpolált árfelület (IDW) és háromdimenziós értékdomborzat (2D csúcs–völgy annotációval, 3D felülettel, metrikaválasztóval).

## MIÉRT fontos?

Egy lakás árát a belső jellemzők mellett az határozza meg, *mi érhető el tőle gyalogosan*, és *mi terheli a közvetlen környezetében*. A két hatás iránya ellentétes lehet: ugyanaz a vasúti infrastruktúra egyrészt elérhetőségi előny (gyors eljutás a városközpontba), másrészt környezeti teher (zaj, rezgés) — a kettőt ezért külön-külön, eltérő mérési logikával kell megfogni, mielőtt egy modellben együtt szerepeltetnénk őket.

## HOGYAN mérünk? Küszöbök kontra folytonos távolság

A mérés két elvet kever. A **környezeti terhelést sávosan** mérjük: a vágánytól mért légvonali távolságot rögzített küszöbökkel (0–150–300–500–1000–2000 méter) osztjuk sávokra, mert a zaj és a rezgés a forrástól távolodva erősen, nem egyenletesen csillapodik — a küszöbök a csillapodási tartományokhoz igazodnak. A **gyalogos elérhetőséget szintén küszöbösen** mérjük, de hálózaton: 1,25 m/s sétatempóval a 375 méter 5 perc, a 750 méter 10 perc, az 1125 méter 15 perc sétatáv — ezek az izokrón-sávok, mert a mindennapi döntések („elérhető-e gyalog?") valójában küszöbdöntések. Emellett mindkét dimenzióban rendelkezésre áll a **folytonos** mérték is (méterben), mert a későbbi modellek a folytonos alakot használják.

## Mi a referencia elve és a döntési szabály?

Minden sávhatást egy **referenciához** viszonyítunk: az a legtávolabbi, még kellően népes (legalább 20 megfigyelésű) sáv, amely a terheléstől gyakorlatilag függetlennek tekinthető — az izokrónoknál ez a 1125 méteren kívüli zóna. A sávok relatív helyzetét így a referencia-kontraszt adja. A fejezet minden számát a közös elemző modul térbeli méréseiből (gyorsítótárból) veszi: a notebook nem számol újra, csak megjelenít.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os; sys.path.insert(0, '..')
from ingatlan_tdk._utils import *
from ingatlan_tdk import analyses as A
setup_plotly()
AREA = os.environ.get('TDK_ACTIVE_AREA', 'kobanya')
df = load_szamitott_master(); p = A.pontos_elado()


In [ ]:
import plotly.express as px
# A terület neve és fejléce dinamikusan a konfigurációból
from IPython.display import display, HTML, Markdown

area_name = get_area_metadata(os.environ.get('TDK_ACTIVE_AREA', 'kobanya'))['name']
display(HTML(render_area_header_html(area=AREA, df=df)))
print(f"Aktív terület: {area_name}  |  teljes minta: {len(df):,} sor  |  elemzési minta (pontos, eladó): {len(p):,} sor".replace(',', ' '))


## 1. Immissziós sávok és gyalogos izokrónok — a mérési táblák

**MIT látunk?** Két táblázatot. Az immissziós tábla a rögzített 0-150-300-500-1000-2000 méteres sávok elemszámait mutatja (hány lakás esik az egyes vágánytávolság-sávokba). Az izokrón-tábla hálózatonként (vasút, metró, villamos, kötöttpálya-minimum) azt, hogy hány lakásból érhető el az adott megálló 5, 10, illetve 15 perc sétával, továbbá a folytonos hálózati távolságok kitöltöttségét és mediánját.

**MIÉRT ezek a küszöbök?** Az immissziós küszöbök a zaj- és rezgéscsillapodás tartományait követik: 150 méteren belül közvetlen, 150–300 méter között erős, 300–500 méter között mérsékelt, 500–1000 és 1000–2000 méter között gyengülő, 2000 méteren túl elhanyagolható terheléssel számolunk. Az izokrón-küszöbök a sétaidőből adódnak: 75 méter/perc (1,25 m/s) mellett a 375/750/1125 méter éppen az 5/10/15 perces gyűrű.

**HOGYAN?** A sávok képzése a feldolgozó láncban, a gyorsítótárban történt; a notebook csak megjeleníti az eredményeket, oszlopdiagramokkal kiegészítve.

**Döntési szabály:** a sávhatások értelmezése mindig a referencia-kontraszt alapján történik (a legtávolabbi, legalább 20 elemű sáv, illetve a 1125 méteren kívüli zóna a viszonyítási alap); a folytonos méter-távolság a későbbi modellek bemenete.


In [ ]:
# 1. Térbeli mérések a cache-ből (A.spatial_metrics) + KPI-k
S = A.spatial_metrics()

vasut15 = int(S['izokronok'].loc[S['izokronok']['halozat'] == 'vasut', '15p_N'].iloc[0])
kp15 = int(S['izokronok'].loc[S['izokronok']['halozat'] == 'kotottpalya', '15p_N'].iloc[0])
poi15_mean = float(S['poi'].loc[S['poi']['poi'] == 'poi_15p_count', 'atlag'].iloc[0])
sav1 = int(S['immisszios_savok']['N'].iloc[0])
med_nm = float(p['nm_ar_huf'].median())
n_p = len(p)

kpi2 = [
    ("Elemzési minta", f"{n_p} db", "Pontos koordinátás eladó lakások", "#1e3a8a"),
    ("Vasút 15p séta", f"{vasut15} db", f"{vasut15/n_p*100:.1f}% lefedettség", "#ec4899"),
    ("Kötöttpálya 15p séta", f"{kp15} db", f"{kp15/n_p*100:.1f}% lefedettség", "#2563eb"),
    ("Átlagos POI 15p", f"{poi15_mean:.1f} db", "Hálózati 15-perces ellátottság", "#059669"),
    ("<150 m immisszió", f"{sav1} db", "Közvetlen vágányközeli sáv", "#dc2626"),
    ("Medián ár/m² (pontos)", fmt_huf(med_nm), "Elemzési minta", "#7c3aed"),
]
display(HTML(kpi_grid_html(kpi2)))

iso = S['izokronok'].copy()
iso.columns = ['Hálózat', '5p (N)', '10p (N)', '15p (N)', 'Folytonos kitöltött', 'Medián táv (m)']
display(HTML("<b>Gyalogos izokrónok (0-375 / 375-750 / 750-1125 m, 1,25 m/s):</b><br>"
             + iso.to_html(classes='table table-bordered table-striped', index=False)))

sav = S['immisszios_savok'].copy()
sav.columns = ['Immissziós sáv', 'N']
display(HTML("<b>Immissziós sávok (légvonali vágánytávolság, 0-150-300-500-1000-2000 m):</b><br>"
             + sav.to_html(classes='table table-bordered table-striped', index=False)))

# Oszlopdiagramok
iso_long = S['izokronok'].melt(id_vars=['halozat'], value_vars=['5p_N', '10p_N', '15p_N'],
                              var_name='izokrón', value_name='N')
iso_long['izokrón'] = iso_long['izokrón'].map({'5p_N': '0-375 m (5 perc)',
                                               '10p_N': '375-750 m (10 perc)',
                                               '15p_N': '750-1125 m (15 perc)'})
fig_iso = px.bar(iso_long, x='halozat', y='N', color='izokrón', barmode='group',
                 title='Gyalogos elérhetőség hálózatonként (lakásszám a sávokban)',
                 labels={'halozat': 'Hálózat', 'N': 'Lakások száma'}, template=PLOTLY_TEMPLATE, height=420)
fig_iso.show()

fig_sav = px.bar(sav, x='Immissziós sáv', y='N', color='N', color_continuous_scale='OrRd', text='N',
                 title='Immissziós sávok elemszámai (elemzési minta)',
                 labels={'N': 'Lakások száma'}, template=PLOTLY_TEMPLATE, height=400)
fig_sav.update_layout(coloraxis_showscale=False)
fig_sav.show()

print(f"Az elemzési minta {n_p} lakásából {vasut15} ({vasut15/n_p*100:.1f}%) éri el a legközelebbi vasútállomást "
      f"15 perc sétán belül; a közvetlen immissziós sávba (<150 m) {sav1} lakás esik.")


## 2. Szolgáltatási ellátottság (POI) a 15 perces körzetben

**MIT mérünk?** Hány darab szolgáltatás (üzlet, rendelő, gyógyszertár, iskola, óvoda, közlekedési pont stb.) érhető el gyalogosan 375, 750, illetve 1125 méteres hálózati sávon belül.

**MIÉRT?** A „15 perces város" gondolata szerint az életminőséget és a lokáció értékét az határozza meg, hogy a napi szükségletek sétatávon belül elérhetők-e. A POI-szám az elérhetőségi előny közvetlen, sávos mérőszáma.

**HOGYAN?** A POI-pontok nyílt térképi (OpenStreetMap-alapú) adatbázisból származnak; a sávonkénti darabszámok a hálózati elérhetőség alapján készültek a feldolgozó láncban, és a gyorsítótárban tárolódnak. A fejezet a sávonkénti átlagokat és a térképi eloszlást mutatja.

**Döntési szabály:** a POI-sávok kumulatívak (a 15 perces szám tartalmazza a közelebbieket is); az értelmezés az átlagos ellátottság és a területen belüli egyenlőtlenség együttes leírása.


In [ ]:
# 2. POI-statisztika a cache-ből + térkép
poi = S['poi'].copy()
poi.columns = ['POI-sáv', 'Átlagos darabszám']
poi['POI-sáv'] = poi['POI-sáv'].map({'poi_5p_count': '5p (≤375 m)',
                                     'poi_10p_count': '10p (≤750 m)',
                                     'poi_15p_count': '15p (≤1125 m)'})
display(HTML("<b>Átlagos szolgáltatásszám a hálózati sávokban (elemzési minta):</b><br>"
             + poi.to_html(classes='table table-bordered table-striped', index=False)))

fig_poi = px.bar(poi, x='POI-sáv', y='Átlagos darabszám', color='Átlagos darabszám',
                 color_continuous_scale='Greens', text='Átlagos darabszám',
                 title='Átlagos gyalogos szolgáltatás-ellátottság (db)', template=PLOTLY_TEMPLATE, height=380)
fig_poi.update_layout(coloraxis_showscale=False)
fig_poi.show()

# Térkép: 15 perces ellátottság pontonként
pmap = p.dropna(subset=['poi_15p_count', 'nm_ar_huf', 'geokodolt_lat', 'geokodolt_lon'])
if len(pmap):
    fig_poi_map = px.scatter_map(
        pmap, lat='geokodolt_lat', lon='geokodolt_lon', color='poi_15p_count',
        size='nm_ar_huf', hover_name='cim_teljes', map_style='carto-positron',
        color_continuous_scale='Viridis',
        title='Hálózaton elérhető szolgáltatások száma 1125 m-en belül (pontonként)')
    fig_poi_map.update_layout(height=500, margin={"r":0,"t":40,"l":0,"b":0})
    fig_poi_map.show()

at15 = float(poi.loc[poi['POI-sáv'] == '15p (≤1125 m)', 'Átlagos darabszám'].iloc[0])
at5 = float(poi.loc[poi['POI-sáv'] == '5p (≤375 m)', 'Átlagos darabszám'].iloc[0])
print(f"Az elemzési mintában átlagosan {at15:.1f} szolgáltatás érhető el 15 perc sétán belül; "
      f"a közvetlen 5 perces körzetben átlagosan {at5:.1f}.")


## 3. Ponttérkép: a fajlagos árak térbeli eloszlása

**MIT látunk?** Minden pont egy eladó lakás az elemzési mintában; a szín a négyzetméterárat, a pontméret az alapterületet jelzi.

**MIÉRT?** A nyers pontfelhő az első ellenőrzés arra, hogy az ár térben strukturált-e (gócok, gradiensek, sávok menti eltérések), mielőtt bármilyen simítást vagy modellt alkalmaznánk.

**HOGYAN?** A térkép tokenmentes webes alaptérképre épül (carto-positron); a színskála az árak 5–95. percentilisére van kalibrálva, hogy néhány szélsőséges hirdetés ne mossa el a különbségeket.

**Döntési szabály:** a ponttérkép diagnosztika; a következő lépésben a pontonkénti zajt IDW-interpolációval (simított felület) csökkentjük.


In [ ]:
# 3. Ponttérkép (px.scatter_map, carto-positron alaptérkép, token nélkül)
def huf_fmt_2(v):
    return f"{v:,.0f} Ft/m²".replace(',', ' ') if pd.notna(v) else "—"

p_map = p.dropna(subset=['nm_ar_huf', 'geokodolt_lat', 'geokodolt_lon']).copy()
fig1 = px.scatter_map(
    p_map,
    lat='geokodolt_lat', lon='geokodolt_lon',
    color='nm_ar_huf', size='alapterulet_nm',
    hover_name='cim_teljes',
    hover_data={'nm_ar_huf': ':.0f', 'ar_millio_ft': ':.1f', 'alapterulet_nm': True,
                'szobaszam_osszes': True, 'varosresz': True},
    color_continuous_scale='Viridis',
    range_color=[p_map['nm_ar_huf'].quantile(0.05), p_map['nm_ar_huf'].quantile(0.95)],
    zoom=12.2, center=get_map_center(), map_style='carto-positron',
    title='Lakások térbeli elhelyezkedése és négyzetméterára (carto-positron alaptérkép)')
fig1.update_layout(height=550, margin={"r":0,"t":40,"l":0,"b":0})
fig1.show()

print(f"A ponttérkép {len(p_map)} lakást ábrázol; a színskála a mintabeli árak "
      f"{huf_fmt_2(p_map['nm_ar_huf'].quantile(0.05))} és {huf_fmt_2(p_map['nm_ar_huf'].quantile(0.95))} közötti tartományára kalibrált.")


## 4. Interpolált árfelület és értékdomborzat (IDW)

**MIT mérünk?** Két egymásra épülő ábrázolást: (a) az árak IDW-interpolációja (távolsággal fordítottan súlyozott átlag, kitevő = 2) ~75 méteres rácson, vetített koordinátákon — színezett felület, izovonalak és a pontos hirdetési pontok rárakva; (b) az **értékdomborzat**: ugyanarról a rácsról 2D „domborzati térkép" a lokális csúcsok és völgyek annotációjával, valamint 3D felület — három választható metrikával (fajlagos ár, log ár, reziduum).

**MIÉRT?** Az egyedi pontok árai zajosak; az IDW-felület folytonos árfelszínt rajzol, amelyen a magas- és alacsony árú gócok, illetve gradiensek közvetlenül leolvashatók. A domborzat csúcsai és völgyei a lokális értékgócokat jelölik ki. A **reziduum-mérték** (a fizikai kontrollok utáni maradék) a vasúton túli „maradék értékdomborzat": a vasúttól független értékkülönbségeket mutatja — városrész-prémiumokat, lokális értékhegyeket és -völgyeket —, ezért érdekes.

**HOGYAN?** A vetítés lokális méter-koordinátákat használ (a szélességi fok koszinuszával korrigált hosszúság: x = lon·cos(mid_lat)·111320, y = lat·110540), így a tengelyek méretarányosak. A rácspontok a cache-elt pontfelhőből (`A.price_surface()`: geokodolt_lat, geokodolt_lon, nm_ar_huf, log_nm_ar, resid) interpolálódnak — megjelenítési lépés, NEM új elemzés. A csúcsok/völgyek detektálása a rácson történik: minden cellát a 3×3-as környezetével vetünk össze (maximum/minimum-szűrés), és a top 5 csúcsot/völgyet ▲/▼ markerrel és értékkel jelöljük. A metrikaválasztó kliensoldali JavaScript, a statikus HTML-exportban is működik.

**Döntési szabály:** a színskála a 2–98. percentilisre kalibrált (a reziduumnál szimmetrikusan, ±max abszolút értékre); a reziduum a kanonikus fizikai kontrollok (méret, szobaszám, panel, lift, erkély, állapot, emelet, épületkor) utáni OLS-maradék.


In [ ]:
# 4A. IDW-interpolált árfelület a cache-elt pontokból (~75 m rács, power=2) — vetített térkép
# A.price_surface() adja a pontokat (geokodolt_lat, geokodolt_lon, nm_ar_huf, log_nm_ar, resid) —
# ez megjelenítési lépés: a notebook NEM számol új elemzést, csak a cache-elt pontokból interpolál.
import numpy as np
import plotly.graph_objects as go

PS = A.price_surface()
df_ar = PS.dropna(subset=['nm_ar_huf', 'geokodolt_lat', 'geokodolt_lon']).copy()
mid_lat = float(df_ar['geokodolt_lat'].mean())
lon_scale = 111320.0 * np.cos(np.radians(mid_lat))
lat_scale = 110540.0
df_ar['x_proj'] = df_ar['geokodolt_lon'] * lon_scale
df_ar['y_proj'] = df_ar['geokodolt_lat'] * lat_scale
# --- TÉRKÉP-ALAPRÉTEG: OSM-utcavonalak + valódi csempe-térkép (épületek, utcák) ---
import json as _json, math as _math, urllib.request as _urlreq, io as _io, base64 as _b64
_RAW_DIR = os.path.join(os.path.dirname(RESULTS_DIR), 'raw')

def _load_basemap():
    streets, rails, stations = [], [], []
    try:
        sf = os.path.join(_RAW_DIR, f'osm_streets_{AREA}_1250m.json')
        for e in _json.load(open(sf, encoding='utf-8')).get('elements', []):
            g = e.get('geometry')
            if g and len(g) >= 2:
                streets.append([(pt['lon'] * lon_scale, pt['lat'] * lat_scale) for pt in g])
    except Exception:
        pass
    try:
        tf = os.path.join(_RAW_DIR, f'osm_{AREA}_transit_2000m.json')
        for e in _json.load(open(tf, encoding='utf-8')).get('elements', []):
            if e.get('type') == 'way' and e.get('geometry') and len(e['geometry']) >= 2:
                t = e.get('tags', {})
                if t.get('railway') in ('rail', 'light_rail'):
                    rails.append([(pt['lon'] * lon_scale, pt['lat'] * lat_scale) for pt in e['geometry']])
            elif e.get('type') == 'node' and e.get('tags', {}).get('railway') in ('station', 'halt'):
                stations.append((e['lon'] * lon_scale, e['lat'] * lat_scale))
    except Exception:
        pass
    return streets, rails, stations

_streets, _rails, _stations = _load_basemap()

def _street_traces(fig, z=None):
    for seg in _streets:
        kw = dict(x=[a for a, _ in seg], y=[b for _, b in seg], mode='lines',
                  line=dict(color='#cbd5e1', width=1.0), hoverinfo='skip',
                  showlegend=False, meta='utcak')
        if z is not None:
            kw.update(z=[z] * len(seg))
        fig.add_trace(go.Scatter(**kw))
    for seg in _rails:
        kw = dict(x=[a for a, _ in seg], y=[b for _, b in seg], mode='lines',
                  line=dict(color='#64748b', width=2.2), hoverinfo='skip',
                  showlegend=False, meta='vasut')
        if z is not None:
            kw.update(z=[z] * len(seg))
        fig.add_trace(go.Scatter(**kw))
    if _stations:
        kw = dict(x=[a for a, _ in _stations], y=[b for _, b in _stations], mode='markers',
                  marker=dict(symbol='square', size=7, color='#dc2626', line=dict(width=1, color='white')),
                  name='Vasútállomás', meta='allomas', hovertemplate='Állomás<extra></extra>')
        if z is not None:
            kw.update(z=[z] * len(_stations))
        fig.add_trace(go.Scatter(**kw))

def _tile_map():
    from PIL import Image
    lat0, lat1 = float(df_ar['geokodolt_lat'].min()), float(df_ar['geokodolt_lat'].max())
    lon0, lon1 = float(df_ar['geokodolt_lon'].min()), float(df_ar['geokodolt_lon'].max())
    for Z in (16, 15, 14):
        def _d2n(lat, lon, z):
            n = 2 ** z
            return (int((lon + 180.0) / 360.0 * n),
                    int((1.0 - _math.asinh(_math.tan(_math.radians(lat))) / _math.pi) / 2.0 * n))
        def _n2d(xt, yt, z):
            n = 2 ** z
            return (_math.degrees(_math.atan(_math.sinh(_math.pi * (1 - 2 * yt / n)))),
                    xt / n * 360.0 - 180.0)
        x0, y1 = _d2n(lat0, lon0, Z)
        x1, y0 = _d2n(lat1, lon1, Z)
        cols, rows = x1 - x0 + 1, y1 - y0 + 1
        if cols * rows > 30:
            continue
        img = Image.new('RGB', (cols * 256, rows * 256), (240, 240, 240))
        for i, xt in enumerate(range(x0, x1 + 1)):
            for j, yt in enumerate(range(y0, y1 + 1)):
                url = f'https://a.basemaps.cartocdn.com/rastertiles/voyager/{Z}/{xt}/{yt}.png'
                req = _urlreq.Request(url, headers={'User-Agent': 'IFK-TDK-2026-Research/1.0'})
                try:
                    tile = Image.open(_io.BytesIO(_urlreq.urlopen(req, timeout=20).read())).convert('RGB')
                    img.paste(tile, (i * 256, j * 256))
                except Exception:
                    pass
        lat_n, lon_w = _n2d(x0, y0, Z)
        lat_s, _ = _n2d(x0, y1 + 1, Z)
        _, lon_e = _n2d(x1 + 1, y0, Z)
        buf = _io.BytesIO()
        img.save(buf, format='PNG')
        return {'b64': 'data:image/png;base64,' + _b64.b64encode(buf.getvalue()).decode('ascii'),
                'west': lon_w * lon_scale, 'east': lon_e * lon_scale,
                'north': lat_n * lat_scale, 'south': lat_s * lat_scale,
                'rgb': np.asarray(img).astype(float) / 255.0,
                'w_m': (lon_e - lon_w) * lon_scale, 'h_m': (lat_n - lat_s) * lat_scale}
    return None

_tile = _tile_map()
print('Térképréteg:', f"betöltve ({_tile['w_m']/1000:.1f} x {_tile['h_m']/1000:.1f} km)" if _tile else 'csempe nem elérhető — utcavonal-alapréteg marad.')

IDW_GRID_M = 75.0
_pad = 150.0
gx = np.arange(float(df_ar['x_proj'].min()) - _pad,
               float(df_ar['x_proj'].max()) + _pad + IDW_GRID_M / 2, IDW_GRID_M)
gy = np.arange(float(df_ar['y_proj'].min()) - _pad,
               float(df_ar['y_proj'].max()) + _pad + IDW_GRID_M / 2, IDW_GRID_M)
GX, GY = np.meshgrid(gx, gy)
grid_nodes = np.column_stack((GX.ravel(), GY.ravel()))
pts = df_ar[['x_proj', 'y_proj']].to_numpy(dtype=float)
vals = df_ar['nm_ar_huf'].to_numpy(dtype=float)

def _idw_surface(nodes, pts, vals, chunk=512):
    z_flat = np.full(len(nodes), np.nan)
    for i in range(0, len(nodes), chunk):
        d = nodes[i:i + chunk, None, :] - pts[None, :, :]
        dist2 = np.einsum('ijk,ijk->ij', d, d)
        w = 1.0 / np.maximum(dist2, 1e-9)
        z_flat[i:i + chunk] = (w @ vals) / w.sum(axis=1)
    return z_flat

Z = _idw_surface(grid_nodes, pts, vals).reshape(GX.shape)

_z_lo, _z_hi = np.nanpercentile(Z, [2, 98])
fig2_idw = go.Figure()
# színezett felület
fig2_idw.add_trace(go.Contour(
    z=Z, x=gx, y=gy, colorscale='Viridis',
    contours=dict(coloring='heatmap', showlabels=False),
    zmin=_z_lo, zmax=_z_hi, colorbar=dict(title='Ft/m²'),
    hovertemplate='Kelet–Nyugat: %{x:.0f} m<br>Észak–Dél: %{y:.0f} m<br>Interpolált ár: %{z:,.0f} Ft/m²<extra></extra>',
    name='IDW-felület'))
_street_traces(fig2_idw)
if _tile:
    fig2_idw.add_layout_image(dict(source=_tile['b64'], xref='x', yref='y',
                                   x=_tile['west'], y=_tile['north'],
                                   sizex=_tile['w_m'], sizey=_tile['h_m'],
                                   sizing='stretch', layer='below', opacity=0.95))
    fig2_idw.data[0].opacity = 0.5
# izovonalak (a felületre rétegezve)
fig2_idw.add_trace(go.Contour(
    z=Z, x=gx, y=gy,
    contours=dict(coloring='lines', showlabels=False),
    line=dict(width=0.5, color='rgba(15,23,42,0.35)'),
    showscale=False, hoverinfo='skip', zmin=_z_lo, zmax=_z_hi,
    name='Izovonalak'))
# a pontos hirdetési pontok kis pöttyökkel rátéve
fig2_idw.add_trace(go.Scatter(
    x=df_ar['x_proj'], y=df_ar['y_proj'], mode='markers',
    marker=dict(size=4, color='#0f172a', opacity=0.6),
    customdata=np.stack([df_ar['nm_ar_huf'].to_numpy()], axis=-1),
    hovertemplate='Hirdetés: %{customdata[0]:,.0f} Ft/m²<extra></extra>',
    name='Hirdetési pontok (pontos minta)'))
fig2_idw.update_layout(
    title='Interpolált árfelület (IDW, power=2) — az értékdomborzat alapja',
    xaxis_title='Kelet–Nyugat (m, lokális síkvetület)',
    yaxis_title='Észak–Dél (m, lokális síkvetület)',
    height=520, template=PLOTLY_TEMPLATE)
fig2_idw.update_yaxes(scaleanchor='x', scaleratio=1.0)
fig2_idw.show()
print(f"IDW-rács: {len(gx)}x{len(gy)} = {len(grid_nodes):,} rácspont (~75 m), "
      f"{len(df_ar)} cache-elt pontból interpolálva; interpolált tartomány (2–98%): "
      f"{fmt_huf(_z_lo)} – {fmt_huf(_z_hi)}".replace(',', ' '))

# 4B. Értékdomborzat — IDW-rács mindhárom metrikára, lokális csúcsok/völgyek, 2D + 3D, JS-választó
# A három metrika: nm_ar_huf (fajlagos ár), log_nm_ar (log ár), resid (fizikai kontrollok utáni
# reziduum = a vasúton túli „maradék értékdomborzat"). A választó a láthatóságot kapcsolja.
import json
import numpy as np
import plotly.graph_objects as go
from scipy.ndimage import maximum_filter, minimum_filter

ps = A.price_surface().dropna(subset=['geokodolt_lat', 'geokodolt_lon']).copy()
_ter_mid = float(ps['geokodolt_lat'].mean())
_ter_lon_s = 111320.0 * np.cos(np.radians(_ter_mid))
_ter_lat_s = 110540.0
ps['x_proj'] = ps['geokodolt_lon'] * _ter_lon_s
ps['y_proj'] = ps['geokodolt_lat'] * _ter_lat_s

TER_GRID_M = 75.0
_ter_pad = 150.0
ter_x = np.arange(ps['x_proj'].min() - _ter_pad, ps['x_proj'].max() + _ter_pad + TER_GRID_M / 2, TER_GRID_M)
ter_y = np.arange(ps['y_proj'].min() - _ter_pad, ps['y_proj'].max() + _ter_pad + TER_GRID_M / 2, TER_GRID_M)
TER_X, TER_Y = np.meshgrid(ter_x, ter_y)
_ter_nodes = np.column_stack((TER_X.ravel(), TER_Y.ravel()))
_ter_pts = ps[['x_proj', 'y_proj']].to_numpy(dtype=float)

def _ter_idw(zvals, chunk=512):
    out = np.full(len(_ter_nodes), np.nan)
    for i in range(0, len(_ter_nodes), chunk):
        d = _ter_nodes[i:i + chunk, None, :] - _ter_pts[None, :, :]
        d2 = np.einsum('ijk,ijk->ij', d, d)
        w = 1.0 / np.maximum(d2, 1e-9)
        out[i:i + chunk] = (w @ zvals) / w.sum(axis=1)
    return out.reshape(TER_X.shape)

def _ter_extrema(Z, top_n=5, min_sep=300.0):
    """Lokális csúcsok/völgyek: 3x3-as környezeti max/min-szűrés, majd érték szerint
    rendezett, egymástól legalább min_sep távolságra eső top_n pont."""
    picks = {}
    for kind in ('max', 'min'):
        filt = maximum_filter if kind == 'max' else minimum_filter
        filler = -np.inf if kind == 'max' else np.inf
        ext = filt(np.where(np.isfinite(Z), Z, filler), size=3)
        mask = (Z == ext) & np.isfinite(Z)
        mask &= (np.arange(Z.shape[0])[:, None] > 0) & (np.arange(Z.shape[0])[:, None] < Z.shape[0] - 1)
        mask &= (np.arange(Z.shape[1])[None, :] > 0) & (np.arange(Z.shape[1])[None, :] < Z.shape[1] - 1)
        cand = [(float(Z[i, j]), i, j) for i, j in zip(*np.nonzero(mask))]
        cand.sort(key=lambda t: t[0], reverse=(kind == 'max'))
        picked = []
        for v, i, j in cand:
            if all((i - pi) ** 2 + (j - pj) ** 2 >= (min_sep / TER_GRID_M) ** 2 for _, pi, pj in picked):
                picked.append((v, i, j))
            if len(picked) >= top_n:
                break
        picks[kind] = picked
    return picks

def _ter_fmt(m, v):
    return f"{v / 1000:,.0f} eFt".replace(',', ' ') if m == 'nm_ar_huf' else f"{v:.2f}"

_ter_metrics = {
    'nm_ar_huf': {'cim': 'Fajlagos ár (Ft/m²)', 'cb': 'Ft/m²', 'scale': 'Viridis'},
    'log_nm_ar': {'cim': 'Log fajlagos ár (ln)', 'cb': 'ln Ft/m²', 'scale': 'Viridis'},
    'resid': {'cim': 'Reziduum (ln, fizikai kontrollok után)', 'cb': 'ln', 'scale': 'RdBu_r'},
}
ter_top_csucs = None
for m, cfg in _ter_metrics.items():
    z = _ter_idw(ps[m].to_numpy(dtype=float))
    lo, hi = np.nanpercentile(z, [2, 98])
    if m == 'resid':
        lim = float(max(abs(lo), abs(hi)))
        lo, hi = -lim, lim
    cfg['z'] = z; cfg['lo'] = float(lo); cfg['hi'] = float(hi)
    if m == 'nm_ar_huf':
        ex0 = _ter_extrema(z)
        ter_top_csucs = float(ex0['max'][0][0]) if ex0['max'] else float('nan')

# --- 2D domborzati térkép: színezett felület + kontúrok + ▲/▼ csúcs/völgy jelölők + pontok ---
fig_ter2 = go.Figure()
_first = True
for m, cfg in _ter_metrics.items():
    z, lo, hi = cfg['z'], cfg['lo'], cfg['hi']
    fig_ter2.add_trace(go.Contour(
        z=z, x=ter_x, y=ter_y, colorscale=cfg['scale'],
        contours=dict(coloring='heatmap', showlabels=False),
        zmin=lo, zmax=hi, colorbar=dict(title=cfg['cb']),
        hovertemplate='x: %{x:.0f} m<br>y: %{y:.0f} m<br>' + cfg['cim'] + ': %{z}<extra></extra>',
        name=cfg['cim'], meta=m, visible=_first))
    fig_ter2.add_trace(go.Contour(
        z=z, x=ter_x, y=ter_y,
        contours=dict(coloring='lines', showlabels=False),
        line=dict(width=0.5, color='rgba(15,23,42,0.35)'),
        showscale=False, hoverinfo='skip', zmin=lo, zmax=hi,
        name=cfg['cim'], meta=m, visible=_first, showlegend=False))
    ex = _ter_extrema(z)
    for kind, mark, colr, tpos in [('max', '▲', '#dc2626', 'top center'), ('min', '▼', '#2563eb', 'bottom center')]:
        xs = [ter_x[j] for _, i, j in ex[kind]]
        ys = [ter_y[i] for _, i, j in ex[kind]]
        vs = [_ter_fmt(m, v) for v, _, _ in ex[kind]]
        fig_ter2.add_trace(go.Scatter(
            x=xs, y=ys, mode='markers+text', text=vs, textposition=tpos,
            textfont=dict(size=10, color=colr),
            marker=dict(symbol='triangle-up' if kind == 'max' else 'triangle-down',
                        size=13, color=colr, line=dict(width=1, color='white')),
            hovertemplate='%{text}<extra></extra>',
            name=f'{cfg["cim"]} — csúcs' if kind == 'max' else f'{cfg["cim"]} — völgy',
            meta=f'{m} · {kind}', visible=_first, showlegend=False))
    _first = False
_street_traces(fig_ter2)
if _tile:
    fig_ter2.add_layout_image(dict(source=_tile['b64'], xref='x', yref='y',
                                   x=_tile['west'], y=_tile['north'],
                                   sizex=_tile['w_m'], sizey=_tile['h_m'],
                                   sizing='stretch', layer='below', opacity=0.95))
    for _tr in fig_ter2.data:
        if getattr(_tr, 'meta', None) in _ter_metrics and _tr.type == 'contour' and _tr.contours.coloring == 'heatmap':
            _tr.opacity = 0.5
fig_ter2.add_trace(go.Scatter(
    x=ps['x_proj'], y=ps['y_proj'], mode='markers',
    marker=dict(size=4, color='#334155', opacity=0.5),
    hovertemplate='Hirdetési pont<extra></extra>',
    name='Hirdetési pontok', meta='pontok'))
fig_ter2.update_layout(
    title='Értékdomborzat 2D — IDW-felület, lokális csúcsok (▲) és völgyek (▼)',
    xaxis_title='Kelet–Nyugat (m, lokális síkvetület)',
    yaxis_title='Észak–Dél (m, lokális síkvetület)',
    height=520, template=PLOTLY_TEMPLATE)
fig_ter2.update_yaxes(scaleanchor='x', scaleratio=1.0)

# --- 3D felület ugyanarról a rácsról ---
fig_ter3 = go.Figure()
_first = True
for m, cfg in _ter_metrics.items():
    fig_ter3.add_trace(go.Surface(
        z=cfg['z'], x=ter_x, y=ter_y, colorscale=cfg['scale'],
        cmin=cfg['lo'], cmax=cfg['hi'], colorbar=dict(title=cfg['cb']),
        opacity=0.95, name=cfg['cim'], meta=m, visible=_first, showscale=_first))
    _first = False
_street_z = min(cfg['lo'] for cfg in _ter_metrics.values()) - 0.02
if _tile:
    _fx = np.linspace(_tile['west'], _tile['east'], _tile['rgb'].shape[1])
    _fy = np.linspace(_tile['north'], _tile['south'], _tile['rgb'].shape[0])
    _FX, _FY = np.meshgrid(_fx, _fy)
    fig_ter3.add_trace(go.Surface(
        z=np.full_like(_FX, _street_z), x=_FX, y=_FY,
        surfacecolor=_tile['rgb'], showscale=False, opacity=0.97,
        name='Térkép (épületek + utcák)', meta='terkep'))
fig_ter3.update_layout(
    title='Értékdomborzat 3D — IDW-felület (go.Surface)',
    scene=dict(xaxis_title='Kelet–Nyugat (m)', yaxis_title='Észak–Dél (m)',
               zaxis_title='Érték', aspectmode='manual',
               aspectratio=dict(x=1, y=1, z=0.35),
               camera=dict(eye=dict(x=1.35, y=1.35, z=0.85))),
    height=560, template=PLOTLY_TEMPLATE, margin=dict(l=0, r=0, t=40, b=0))

# --- JS-választó: a select a láthatóságot kapcsolja (a statikus HTML-exportban is működik) ---
_ter_opts = ''.join(f'<option value="{m}" data-val="{m}">{cfg["cim"]}</option>' for m, cfg in _ter_metrics.items())
_ter_html = f"""
<div style="background:#f8fafc;border:1px solid #cbd5e1;border-radius:12px;padding:16px;margin:16px 0;">
<h3 style="margin:0 0 10px 0;">Értékdomborzat — metrikaválasztó (kliensoldali, statikus HTML-ben is működik)</h3>
<div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:12px;">
<label>Domborzat-mutató:
  <select id="nb02_ter_sel" style="padding:6px;">{_ter_opts}</select>
</label>
</div>
<div id="nb02_ter_2d" style="width:100%;height:520px;"></div>
<div id="nb02_ter_3d" style="width:100%;height:560px;"></div>
</div>
<script type="application/json" id="nb02_ter_2d_data">{fig_ter2.to_json()}</script>
<script type="application/json" id="nb02_ter_3d_data">{fig_ter3.to_json()}</script>
<script>
(function() {{
  var sel = document.getElementById('nb02_ter_sel');
  function apply() {{
    var m = sel.value;
    ['nb02_ter_2d', 'nb02_ter_3d'].forEach(function (gid) {{
      var gd = document.getElementById(gid);
      if (!gd || !gd.data) return;
      var vis = gd.data.map(function (t) {{
        return t.meta === m || (t.meta || '').indexOf(m + ' ·') === 0 ||
             t.meta === 'utcak' || t.meta === 'vasut' || t.meta === 'allomas' ||
             t.meta === 'terkep' || t.name === 'Vasútállomás';
      }});
      if (gid === 'nb02_ter_2d') {{
        gd.data.forEach(function (t, i) {{ if (t.name === 'Hirdetési pontok') vis[i] = true; }});
      }}
      Plotly.restyle(gd, {{visible: vis}});
    }});
  }}
  function ready() {{
    if (!window.Plotly) {{ setTimeout(ready, 200); return; }}
    var s2 = JSON.parse(document.getElementById('nb02_ter_2d_data').textContent);
    var s3 = JSON.parse(document.getElementById('nb02_ter_3d_data').textContent);
    Plotly.newPlot('nb02_ter_2d', s2.data, s2.layout, {{displaylogo: false}});
    Plotly.newPlot('nb02_ter_3d', s3.data, s3.layout, {{displaylogo: false}});
    apply();
  }}
  sel.addEventListener('change', apply);
  ready();
}})();
</script>"""
display(HTML(_ter_html))
print(f"Értékdomborzat: {len(ter_x)}x{len(ter_y)} = {len(_ter_nodes):,} rácspont (~75 m), {len(ps)} cache-elt pontból; "
      f"a legmagasabb lokális árcsúcs {fmt_huf(ter_top_csucs) if pd.notna(ter_top_csucs) else '—'}. "
      f"A választó a 2D és 3D ábra láthatóságát kapcsolja (nm_ar_huf / log_nm_ar / resid).".replace(',', ' '))


## 5. Hálózati távolságok és a kerülőfaktor

**MIT mérünk?** Azt, hogy a valódi gyalogos útvonal hossza mennyivel haladja meg a légvonali távolságot. A **kerülőfaktor** a hálózati (utcahálózaton mért legrövidebb) út és a légvonali távolság hányadosa.

**MIÉRT?** A légvonali távolság szisztematikusan alulbecsüli a valós gyaloglást: az utcahálózat kerülői, a kényszerű átkelők és a zsákutcák miatt a tényleges sétaút jellemzően másfél-kétszeresére nyúlik. Ezért minden elérhetőségi elemzés hálózati távolságokra épül, és a kerülőfaktor a feldolgozó lánc állandó ellenőrző mutatója.

**HOGYAN?** A hálózati távolságok az utcahálózat-gráfon futó legrövidebb út kereséséből származnak (a feldolgozó láncban, gyorsítótárban); a notebook a medián kerülőfaktort és a hálózati távolságok mediánjait mutatja meg hálózatonként.

**Döntési szabály:** ha a kerülőfaktor szélsőséges (egynél kisebb vagy nagyon nagy), az adathibára utal; a jellemző 1,5–2 közötti érték az egészséges városi úthálózat jele.


In [ ]:
# 5. Kerülőfaktor és hálózati távolságok — számítás a cache adataiból, f-string kimenet
detour_rows = p.dropna(subset=['tavolsag_vasut_halozati_m', 'tavolsag_vasut_m'])
detour = (detour_rows['tavolsag_vasut_halozati_m'] / detour_rows['tavolsag_vasut_m'].replace(0, np.nan)).dropna()
detour_med = float(detour.median())

iso2 = S['izokronok']
net_lines = []
for _, r in iso2.iterrows():
    net_lines.append(f"{r['halozat']}: 15 perces körzetben {int(r['15p_N'])} lakás, "
                     f"folytonos távolság mediánja {r['median_tav_m']:.0f} m "
                     f"({int(r['folytonos_kitoltott'])} kitöltött érték)")
print("Hálózati távolságok hálózatonként: " + "; ".join(net_lines))
print(f"A kerülőfaktor (hálózati út / légvonali távolság) mediánja az elemzési mintán: "
      f"{detour_med:.2f} — azaz a valós sétaút mediánban közel {detour_med:.2f}-szerese "
      f"a mértani távolságnak (n = {len(detour)} megfigyelés).")


## Összefoglalás

A fejezet térbeli méréseit egyetlen mondatfolyamba rendezzük: minden szám a közös elemző modul gyorsítótárából, futásidőben illesztve.


In [ ]:
# Záró összefoglaló — minden szám a cache-ből, f-stringgel (nincs hardkódolt érték)
def _n2(v):
    return f"{int(v):,}".replace(',', ' ')

iso3 = S['izokronok']; sav3 = S['immisszios_savok']; poi3 = S['poi']
poi15 = float(poi3.loc[poi3['poi'] == 'poi_15p_count', 'atlag'].iloc[0])
vas = iso3.loc[iso3['halozat'] == 'vasut']
kp = iso3.loc[iso3['halozat'] == 'kotottpalya']

print(
    f"Az elemzési minta {_n2(len(p))} lakásából {_n2(int(vas['15p_N'].iloc[0]))} lakás éri el a legközelebbi vasútállomást "
    f"15 perc sétán belül, a kötöttpályás hálózat valamely megállóját pedig {_n2(int(kp['15p_N'].iloc[0]))} lakás. "
    f"A közvetlen immissziós sávban (<150 m a vágánytól) {_n2(int(sav3['N'].iloc[0]))} lakás fekszik. "
    f"A gyalogos körzetben átlagosan {poi15:.1f} szolgáltatás érhető el 15 percen belül. "
    f"A hálózati sétaút a légvonalihoz képest mediánban {detour_med:.2f}-szeres (kerülőfaktor). "
    f"A térbeli árszerkezet az IDW-rácson {_n2(len(grid_nodes))} rácspontból épül fel, "
    f"a domborzat legmagasabb lokális árcsúcsa {fmt_huf(ter_top_csucs) if pd.notna(ter_top_csucs) else '—'}."
)
